# Historical Selling Opportunities (Tops) Analysis

Looking back at the last 5 years to identify:
1. What were the best selling opportunities (in hindsight)?
2. What were SOPR/STH-SOPR/MVRV values at those times?
3. What patterns can we learn for exits?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Historical Tops Analysis 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
nupl = pd.read_parquet(DATA_DIR / "nupl.parquet").rename(columns={"value": "nupl"}).set_index("time") if (DATA_DIR / "nupl.parquet").exists() else None

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner')
if nupl is not None:
    df = df.join(nupl, how='inner')
df = df.sort_index()

# Last 5 years
df = df[df.index >= '2020-01-01'].copy()
df = df.dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Calculate forward returns (negative = good sell)
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1
df['fwd_180d'] = df['price'].shift(-180) / df['price'] - 1
df['fwd_365d'] = df['price'].shift(-365) / df['price'] - 1

# Distance from ATH and local highs
df['ath'] = df['price'].cummax()
df['at_ath'] = df['price'] == df['ath']
df['pct_from_ath'] = (df['price'] - df['ath']) / df['ath']

print("Calculated metrics")

---
## 1. Known Major Tops

In [ ]:
# Known major selling opportunities (hindsight)
major_tops = [
    ('2020-02-13', 'Pre-COVID High', 'Before March crash'),
    ('2020-08-17', 'Summer 2020 High', 'Before consolidation'),
    ('2021-01-08', 'Jan 2021 Peak', 'First leg of bull'),
    ('2021-02-21', 'Feb 2021 Peak', 'Pre-correction'),
    ('2021-04-14', 'April ATH', 'Coinbase listing peak'),
    ('2021-05-10', 'Pre-Crash High', 'Before China ban crash'),
    ('2021-09-07', 'Sept Local High', 'El Salvador pump'),
    ('2021-10-20', 'Oct 2021 High', 'Pre-ATH run'),
    ('2021-11-10', 'Cycle ATH', '$69k all-time high'),
    ('2022-03-28', 'Bear Rally 1', 'First bear market rally'),
    ('2022-08-15', 'Bear Rally 2', 'Summer 2022 relief rally'),
    ('2022-11-05', 'Pre-FTX High', 'Before FTX collapse'),
    ('2023-04-14', 'Spring 2023 High', 'Before summer correction'),
    ('2023-07-13', 'July 2023 High', 'Local top'),
    ('2023-12-08', 'Dec 2023 High', 'Pre-ETF speculation peak'),
    ('2024-03-14', 'New ATH', '$73k peak'),
    ('2024-06-07', 'June 2024 High', 'Local high'),
    ('2024-12-17', 'Dec 2024 ATH', '$108k all-time high'),
]

print(f"Analyzing {len(major_tops)} known tops")

In [ ]:
# Analyze each top
print("MAJOR TOPS - METRIC VALUES")
print("="*150)
print(f"{'Date':<12} {'Event':<20} {'Price':>10} {'SOPR':>8} {'STH-SOPR':>10} {'MVRV':>8} {'30d Fwd':>10} {'90d Fwd':>10} {'180d Fwd':>10}")
print("-"*150)

top_data = []

for date_str, name, desc in major_tops:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        actual_date = df.index[idx]
        
        top_data.append({
            'date': actual_date,
            'name': name,
            'price': row['price'],
            'sopr': row['sopr'],
            'sth_sopr': row['sopr_sth'],
            'mvrv': row['mvrv'],
            'fwd_30d': row['fwd_30d'],
            'fwd_90d': row['fwd_90d'],
            'fwd_180d': row['fwd_180d'],
        })
        
        fwd30 = f"{row['fwd_30d']*100:+.0f}%" if pd.notna(row['fwd_30d']) else 'N/A'
        fwd90 = f"{row['fwd_90d']*100:+.0f}%" if pd.notna(row['fwd_90d']) else 'N/A'
        fwd180 = f"{row['fwd_180d']*100:+.0f}%" if pd.notna(row['fwd_180d']) else 'N/A'
        
        print(f"{actual_date.strftime('%Y-%m-%d'):<12} {name:<20} ${row['price']:>9,.0f} {row['sopr']:>8.3f} {row['sopr_sth']:>10.3f} {row['mvrv']:>8.2f} {fwd30:>10} {fwd90:>10} {fwd180:>10}")
    except Exception as e:
        print(f"{date_str:<12} {name:<20} Error: {e}")

tops_df = pd.DataFrame(top_data)

In [ ]:
# Summary statistics for tops
print("\n" + "="*70)
print("METRIC VALUES AT MAJOR TOPS - SUMMARY")
print("="*70)

print(f"\nSOPR at tops:")
print(f"  Min:    {tops_df['sopr'].min():.3f}")
print(f"  Max:    {tops_df['sopr'].max():.3f}")
print(f"  Mean:   {tops_df['sopr'].mean():.3f}")
print(f"  Median: {tops_df['sopr'].median():.3f}")

print(f"\nSTH-SOPR at tops:")
print(f"  Min:    {tops_df['sth_sopr'].min():.3f}")
print(f"  Max:    {tops_df['sth_sopr'].max():.3f}")
print(f"  Mean:   {tops_df['sth_sopr'].mean():.3f}")
print(f"  Median: {tops_df['sth_sopr'].median():.3f}")

print(f"\nMVRV at tops:")
print(f"  Min:    {tops_df['mvrv'].min():.2f}")
print(f"  Max:    {tops_df['mvrv'].max():.2f}")
print(f"  Mean:   {tops_df['mvrv'].mean():.2f}")
print(f"  Median: {tops_df['mvrv'].median():.2f}")

print(f"\n% of tops where SOPR > 1:     {(tops_df['sopr'] > 1).mean()*100:.0f}%")
print(f"% of tops where SOPR > 1.02:  {(tops_df['sopr'] > 1.02).mean()*100:.0f}%")
print(f"% of tops where SOPR > 1.05:  {(tops_df['sopr'] > 1.05).mean()*100:.0f}%")
print(f"% of tops where STH-SOPR > 1: {(tops_df['sth_sopr'] > 1).mean()*100:.0f}%")
print(f"% of tops where MVRV > 2.0:   {(tops_df['mvrv'] > 2.0).mean()*100:.0f}%")
print(f"% of tops where MVRV > 2.5:   {(tops_df['mvrv'] > 2.5).mean()*100:.0f}%")
print(f"% of tops where MVRV > 3.0:   {(tops_df['mvrv'] > 3.0).mean()*100:.0f}%")

---
## 2. Visualize Price with Exit Signals

In [ ]:
# Create comprehensive chart
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=('BTC Price (Log)', 'SOPR', 'STH-SOPR', 'MVRV')
)

# Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)

# Mark tops on price
for _, t in tops_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[t['date']], y=[t['price']], 
        mode='markers', marker=dict(size=12, color='red', symbol='triangle-down'),
        name=t['name'], showlegend=False,
        hovertext=f"{t['name']}<br>SOPR: {t['sopr']:.3f}<br>STH: {t['sth_sopr']:.3f}<br>MVRV: {t['mvrv']:.2f}"
    ), row=1, col=1)

# SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR', line=dict(color='blue')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=2, col=1)
fig.add_hline(y=1.02, line_dash='dash', line_color='orange', row=2, col=1)
fig.add_hline(y=1.05, line_dash='dash', line_color='red', row=2, col=1)

# STH-SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_sth'], name='STH-SOPR', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=3, col=1)
fig.add_hline(y=1.02, line_dash='dash', line_color='orange', row=3, col=1)
fig.add_hline(y=1.05, line_dash='dash', line_color='red', row=3, col=1)

# MVRV
fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV', line=dict(color='green')), row=4, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=4, col=1)
fig.add_hline(y=2, line_dash='dash', line_color='orange', row=4, col=1)
fig.add_hline(y=2.5, line_dash='dash', line_color='red', row=4, col=1)
fig.add_hline(y=3, line_dash='dash', line_color='darkred', row=4, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=1000, title='BTC Price vs On-Chain Metrics at Tops', showlegend=False)
fig.show()

---
## 3. Forward Returns by Metric Level (Exit Signals)

In [ ]:
# Forward returns when metrics are elevated (potential sell signals)
print("FORWARD RETURNS BY METRIC LEVEL (EXIT SIGNALS)")
print("="*100)

print(f"\n{'Condition':<25} {'Avg 30d':>12} {'Avg 90d':>12} {'Avg 180d':>12} {'Days':>10} {'% Time':>10}")
print("-"*100)

conditions = [
    ('All days', df['sopr'] > 0),
    
    # SOPR sell signals
    ('SOPR > 1', df['sopr'] > 1),
    ('SOPR > 1.02', df['sopr'] > 1.02),
    ('SOPR > 1.05', df['sopr'] > 1.05),
    ('SOPR > 1.08', df['sopr'] > 1.08),
    ('SOPR > 1.10', df['sopr'] > 1.10),
    
    # STH-SOPR sell signals
    ('STH-SOPR > 1', df['sopr_sth'] > 1),
    ('STH-SOPR > 1.02', df['sopr_sth'] > 1.02),
    ('STH-SOPR > 1.05', df['sopr_sth'] > 1.05),
    ('STH-SOPR > 1.08', df['sopr_sth'] > 1.08),
    ('STH-SOPR > 1.10', df['sopr_sth'] > 1.10),
    
    # MVRV sell signals
    ('MVRV > 1.5', df['mvrv'] > 1.5),
    ('MVRV > 2.0', df['mvrv'] > 2.0),
    ('MVRV > 2.5', df['mvrv'] > 2.5),
    ('MVRV > 3.0', df['mvrv'] > 3.0),
    
    # Combined
    ('SOPR>1.02 & MVRV>2', (df['sopr'] > 1.02) & (df['mvrv'] > 2)),
    ('STH>1.02 & MVRV>2', (df['sopr_sth'] > 1.02) & (df['mvrv'] > 2)),
    ('SOPR>1.05 & MVRV>2.5', (df['sopr'] > 1.05) & (df['mvrv'] > 2.5)),
]

for name, cond in conditions:
    subset = df[cond]
    if len(subset) > 10:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        avg_180 = subset['fwd_180d'].mean() * 100
        pct_time = len(subset) / len(df) * 100
        print(f"{name:<25} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {avg_180:>+11.1f}% {len(subset):>10} {pct_time:>9.1f}%")

In [ ]:
# Best exit signals (most negative forward returns)
print("\n" + "="*70)
print("BEST EXIT SIGNALS (Most Negative 90-Day Forward Returns)")
print("="*70)

exit_signals = [
    ('SOPR > 1.02', df['sopr'] > 1.02),
    ('SOPR > 1.05', df['sopr'] > 1.05),
    ('SOPR > 1.08', df['sopr'] > 1.08),
    ('STH-SOPR > 1.02', df['sopr_sth'] > 1.02),
    ('STH-SOPR > 1.05', df['sopr_sth'] > 1.05),
    ('STH-SOPR > 1.08', df['sopr_sth'] > 1.08),
    ('MVRV > 2.0', df['mvrv'] > 2.0),
    ('MVRV > 2.5', df['mvrv'] > 2.5),
    ('MVRV > 3.0', df['mvrv'] > 3.0),
    ('SOPR>1.02 & MVRV>2', (df['sopr'] > 1.02) & (df['mvrv'] > 2)),
    ('SOPR>1.05 & MVRV>2.5', (df['sopr'] > 1.05) & (df['mvrv'] > 2.5)),
]

exit_results = []
for name, cond in exit_signals:
    subset = df[cond]
    if len(subset) > 10:
        exit_results.append({
            'name': name,
            'avg_90d': subset['fwd_90d'].mean() * 100,
            'count': len(subset),
            'pct_time': len(subset) / len(df) * 100
        })

exit_results = sorted(exit_results, key=lambda x: x['avg_90d'])

print(f"\n{'Rank':<6} {'Signal':<30} {'Avg 90d Fwd':>15} {'% Time':>10}")
print("-"*70)
for i, r in enumerate(exit_results):
    quality = '⭐' if r['avg_90d'] < -5 else ''
    print(f"{i+1:<6} {r['name']:<30} {r['avg_90d']:>+14.1f}% {r['pct_time']:>9.1f}% {quality}")

---
## 4. Tops Ranked by Forward Loss

In [ ]:
# Rank tops by how bad the subsequent drawdown was
print("TOPS RANKED BY 90-DAY FORWARD LOSS")
print("="*120)

valid_tops = tops_df[tops_df['fwd_90d'].notna()].copy()
valid_tops = valid_tops.sort_values('fwd_90d', ascending=True)

print(f"\n{'Rank':<6} {'Date':<12} {'Event':<20} {'90d Loss':>12} {'SOPR':>8} {'STH-SOPR':>10} {'MVRV':>8}")
print("-"*90)

for i, (_, row) in enumerate(valid_tops.iterrows()):
    mvrv_flag = '🔴' if row['mvrv'] > 2.5 else ('🟡' if row['mvrv'] > 2 else '')
    print(f"{i+1:<6} {row['date'].strftime('%Y-%m-%d'):<12} {row['name']:<20} {row['fwd_90d']*100:>+11.0f}% {row['sopr']:>8.3f} {row['sth_sopr']:>10.3f} {row['mvrv']:>7.2f} {mvrv_flag}")

In [ ]:
# Signal accuracy at tops
print("\nSIGNAL ACCURACY AT MAJOR TOPS")
print("="*70)

# Only count tops with actual negative forward returns
real_tops = tops_df[tops_df['fwd_90d'] < -0.10]  # At least 10% drop
print(f"\nReal tops (>10% drop in 90d): {len(real_tops)}/{len(tops_df)}")

if len(real_tops) > 0:
    print(f"\nAt real tops:")
    print(f"  SOPR > 1.02:  {(real_tops['sopr'] > 1.02).sum()}/{len(real_tops)} ({(real_tops['sopr'] > 1.02).mean()*100:.0f}%)")
    print(f"  SOPR > 1.05:  {(real_tops['sopr'] > 1.05).sum()}/{len(real_tops)} ({(real_tops['sopr'] > 1.05).mean()*100:.0f}%)")
    print(f"  STH > 1.02:   {(real_tops['sth_sopr'] > 1.02).sum()}/{len(real_tops)} ({(real_tops['sth_sopr'] > 1.02).mean()*100:.0f}%)")
    print(f"  STH > 1.05:   {(real_tops['sth_sopr'] > 1.05).sum()}/{len(real_tops)} ({(real_tops['sth_sopr'] > 1.05).mean()*100:.0f}%)")
    print(f"  MVRV > 2.0:   {(real_tops['mvrv'] > 2.0).sum()}/{len(real_tops)} ({(real_tops['mvrv'] > 2.0).mean()*100:.0f}%)")
    print(f"  MVRV > 2.5:   {(real_tops['mvrv'] > 2.5).sum()}/{len(real_tops)} ({(real_tops['mvrv'] > 2.5).mean()*100:.0f}%)")

---
## 5. MVRV Distribution Analysis

In [ ]:
# MVRV threshold analysis
print("MVRV THRESHOLD ANALYSIS")
print("="*100)
print(f"\n{'MVRV Threshold':<18} {'Days':>8} {'% Time':>10} {'Avg 30d':>12} {'Avg 90d':>12} {'Avg 180d':>12}")
print("-"*100)

for t in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25, 3.5]:
    cond = df['mvrv'] > t
    subset = df[cond]
    if len(subset) > 10:
        pct_time = len(subset) / len(df) * 100
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        avg_180 = subset['fwd_180d'].mean() * 100
        
        print(f"MVRV > {t:<11} {len(subset):>8} {pct_time:>9.1f}% {avg_30:>+11.1f}% {avg_90:>+11.1f}% {avg_180:>+11.1f}%")

---
## 6. Compare Bottoms vs Tops

In [ ]:
# Load bottoms data from previous analysis
major_bottoms = [
    ('2020-03-12', 'COVID Crash'),
    ('2021-05-19', 'China Ban'),
    ('2021-07-20', 'July Bottom'),
    ('2022-06-18', 'June 2022'),
    ('2022-11-21', 'Cycle Bottom'),
    ('2024-08-05', 'Yen Carry'),
]

bottom_metrics = []
for date_str, name in major_bottoms:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        bottom_metrics.append({'type': 'Bottom', 'name': name, 'sopr': row['sopr'], 'sth_sopr': row['sopr_sth'], 'mvrv': row['mvrv']})
    except:
        pass

major_tops_short = [
    ('2021-04-14', 'April ATH'),
    ('2021-11-10', 'Cycle ATH'),
    ('2024-03-14', 'New ATH'),
    ('2024-12-17', 'Dec ATH'),
]

top_metrics = []
for date_str, name in major_tops_short:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        top_metrics.append({'type': 'Top', 'name': name, 'sopr': row['sopr'], 'sth_sopr': row['sopr_sth'], 'mvrv': row['mvrv']})
    except:
        pass

compare_df = pd.DataFrame(bottom_metrics + top_metrics)

print("BOTTOMS vs TOPS COMPARISON")
print("="*80)
print(f"\n{'Metric':<15} {'Bottoms (avg)':>15} {'Tops (avg)':>15} {'Difference':>15}")
print("-"*60)

bottoms = compare_df[compare_df['type'] == 'Bottom']
tops = compare_df[compare_df['type'] == 'Top']

for col in ['sopr', 'sth_sopr', 'mvrv']:
    b_avg = bottoms[col].mean()
    t_avg = tops[col].mean()
    diff = t_avg - b_avg
    print(f"{col.upper():<15} {b_avg:>15.3f} {t_avg:>15.3f} {diff:>+15.3f}")

---
## 7. Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS - TOPS")
print("="*70)

print(f"""
📊 AT MAJOR TOPS:
   • Average SOPR: {tops_df['sopr'].mean():.3f}
   • Average STH-SOPR: {tops_df['sth_sopr'].mean():.3f}
   • Average MVRV: {tops_df['mvrv'].mean():.2f}
   
   • SOPR > 1.02 at {(tops_df['sopr'] > 1.02).mean()*100:.0f}% of tops
   • SOPR > 1.05 at {(tops_df['sopr'] > 1.05).mean()*100:.0f}% of tops
   • MVRV > 2.0 at {(tops_df['mvrv'] > 2.0).mean()*100:.0f}% of tops
   • MVRV > 2.5 at {(tops_df['mvrv'] > 2.5).mean()*100:.0f}% of tops

📉 FORWARD RETURNS (when exit signal active):
   • SOPR > 1.02: avg 90d return = {df[df['sopr'] > 1.02]['fwd_90d'].mean()*100:+.1f}%
   • SOPR > 1.05: avg 90d return = {df[df['sopr'] > 1.05]['fwd_90d'].mean()*100:+.1f}%
   • MVRV > 2.0: avg 90d return = {df[df['mvrv'] > 2.0]['fwd_90d'].mean()*100:+.1f}%
   • MVRV > 2.5: avg 90d return = {df[df['mvrv'] > 2.5]['fwd_90d'].mean()*100:+.1f}%

🔍 SIGNAL FREQUENCY:
   • SOPR > 1.02: {(df['sopr'] > 1.02).mean()*100:.1f}% of days
   • SOPR > 1.05: {(df['sopr'] > 1.05).mean()*100:.1f}% of days
   • MVRV > 2.0: {(df['mvrv'] > 2.0).mean()*100:.1f}% of days
   • MVRV > 2.5: {(df['mvrv'] > 2.5).mean()*100:.1f}% of days

💡 COMPARISON TO BOTTOMS:
   • Bottoms: SOPR ~ 0.95, STH-SOPR ~ 0.95, MVRV ~ low
   • Tops: SOPR ~ {tops_df['sopr'].mean():.2f}, STH-SOPR ~ {tops_df['sth_sopr'].mean():.2f}, MVRV ~ {tops_df['mvrv'].mean():.1f}
""")